# Build depth-following PV-gradient tables

This notebook follows each fitted eddy ellipse through the upper 1000 m. It calculates horizontal ellipse means at every fitted depth, then forms a thickness-weighted eddy-day table by averaging vector components before reconstructing magnitudes and bearings. The calculation is intentionally cached because it is substantially more expensive than the surface-centred method.

In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
import seacofs_tilt_tools as tilt
pd.set_option('display.max_columns', 80)

## Settings

The output directory is separate from the source eddy tables. Existing cache files are replaced only after each complete Parquet file has been written.

In [2]:
MAX_DEPTH_M = 1000.0
OUTPUT_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/pv_gradient_depth_following')
SNAPSHOT_PATH = OUTPUT_ROOT / 'pv_gradient_depth_following_snapshot_0_1000m.parquet'
DEPTH_PATH = OUTPUT_ROOT / 'pv_gradient_depth_following_depth_0_1000m.parquet'
METADATA_PATH = OUTPUT_ROOT / 'pv_gradient_depth_following_metadata.json'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SNAPSHOT_PATH, DEPTH_PATH

(PosixPath('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/pv_gradient_depth_following/pv_gradient_depth_following_snapshot_0_1000m.parquet'),
 PosixPath('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/pv_gradient_depth_following/pv_gradient_depth_following_depth_0_1000m.parquet'))

## Load authoritative snapshot, tilt and vertical-profile tables

In [3]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
vertical = tilt.load_vert(paths, dic_form=False)

required = {'Eddy', 'Day', 'Depth', 'xc', 'yc', 'Rc', 'q11', 'q12', 'q22', 'w'}
missing = required - set(vertical.columns)
if missing:
    raise KeyError(f'Vertical table is missing {sorted(missing)}')
if eddies.duplicated(['Eddy', 'Day']).any():
    raise ValueError('Snapshot table contains duplicate Eddy-Day rows')
if vertical.duplicated(['Eddy', 'Day', 'Depth']).any():
    raise ValueError('Vertical table contains duplicate Eddy-Day-Depth rows')
display(pd.Series({'snapshots': len(eddies), 'eddies': eddies.Eddy.nunique(),
                   'vertical_rows': len(vertical),
                   'vertical_rows_to_1000m': vertical.Depth.abs().le(MAX_DEPTH_M).sum()}))

snapshots                  127426
eddies                       2982
vertical_rows             2824609
vertical_rows_to_1000m    2468392
dtype: int64

## Process depth-following ellipse means

In [ ]:
started = time.perf_counter()
snapshot_df, depth_df = tilt.add_pv_gradient_terms(
    eddies, grid, core_mean=True, depth_following=True,
    vertical=vertical, max_depth_m=MAX_DEPTH_M, progress_every=10000,
)
elapsed_minutes = (time.perf_counter() - started) / 60
print(f'Completed in {elapsed_minutes:.1f} minutes')

Processed 10,000/2,468,392 depth ellipses
Processed 20,000/2,468,392 depth ellipses
Processed 30,000/2,468,392 depth ellipses
Processed 40,000/2,468,392 depth ellipses
Processed 50,000/2,468,392 depth ellipses
Processed 60,000/2,468,392 depth ellipses
Processed 70,000/2,468,392 depth ellipses
Processed 80,000/2,468,392 depth ellipses
Processed 90,000/2,468,392 depth ellipses
Processed 100,000/2,468,392 depth ellipses
Processed 110,000/2,468,392 depth ellipses
Processed 120,000/2,468,392 depth ellipses
Processed 130,000/2,468,392 depth ellipses
Processed 140,000/2,468,392 depth ellipses
Processed 150,000/2,468,392 depth ellipses
Processed 160,000/2,468,392 depth ellipses
Processed 170,000/2,468,392 depth ellipses
Processed 180,000/2,468,392 depth ellipses
Processed 190,000/2,468,392 depth ellipses
Processed 200,000/2,468,392 depth ellipses
Processed 210,000/2,468,392 depth ellipses
Processed 220,000/2,468,392 depth ellipses
Processed 230,000/2,468,392 depth ellipses
Processed 240,000/2,

## Quality and coverage checks

In [ ]:
assert len(snapshot_df) == len(eddies)
assert not snapshot_df.duplicated(['Eddy', 'Day']).any()
assert not depth_df.duplicated(['Eddy', 'Day', 'Depth']).any()
assert depth_df.Depth.between(0, MAX_DEPTH_M).all()
assert np.allclose(snapshot_df.PV_grad_x,
                   snapshot_df.PV_grad_plan_x + snapshot_df.PV_grad_topo_x,
                   equal_nan=True)
assert np.allclose(snapshot_df.PV_grad_y,
                   snapshot_df.PV_grad_plan_y + snapshot_df.PV_grad_topo_y,
                   equal_nan=True)
display(snapshot_df.groupby('Cyc').agg(
    snapshots=('Eddy', 'size'), eddies=('Eddy', 'nunique'),
    valid_gradient=('PV_grad_mag', 'count'),
    median_depth_levels=('PV_depth_n', 'median'),
    median_vertical_coverage=('PV_vertical_coverage_fraction', 'median'),
).round(3))
display(snapshot_df[['PV_depth_n', 'PV_depth_max_m',
                     'PV_vertical_coverage_m',
                     'PV_vertical_coverage_fraction',
                     'PV_mean_Rc_km']].describe().round(3))

## Save reusable cache

In [ ]:
def atomic_parquet(frame, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

atomic_parquet(snapshot_df, SNAPSHOT_PATH)
atomic_parquet(depth_df, DEPTH_PATH)
metadata = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'max_depth_m': MAX_DEPTH_M,
    'method': 'depth-following local ellipse; horizontal mean of completed components; thickness-weighted vertical component mean',
    'snapshot_rows': len(snapshot_df),
    'depth_rows': len(depth_df),
    'snapshot_path': str(SNAPSHOT_PATH),
    'depth_path': str(DEPTH_PATH),
}
temporary_metadata = METADATA_PATH.with_suffix('.json.tmp')
temporary_metadata.write_text(json.dumps(metadata, indent=2))
temporary_metadata.replace(METADATA_PATH)
print(f'Saved {SNAPSHOT_PATH}')
print(f'Saved {DEPTH_PATH}')
print(f'Saved {METADATA_PATH}')